In [32]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


In [33]:
SEARCH_QUERY = "wireless headphones"
NUM_PAGES = 20
OUTPUT_FILE = "amazon_wireless_headphones.csv"

def clean_rating(text):
    if not text:
        return None
    match = re.search(r"\d+\.\d+", text)
    return float(match.group()) if match else None

options = Options()
options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

rows = []

try:
    for page in range(1, NUM_PAGES + 1):
        #url = f"https://www.amazon.com/s?k=wireless+headphones&rh=n%3A21514463011%2Cp_123%3A233043%257C264616%257C325772&dc&crid=37RAF0SVWP0ZZ&qid=1776023787&rnid=85457740011&sprefix=%2Caps%2C177&ref=sr_nr_p_123_4&ds=v1%3AaS7xNxv3ETbq0rBMp0a1QOxUuJUGO4XFffw2gLNIQOY"
        url = f"https://www.amazon.com/s?k={SEARCH_QUERY.replace(' ', '+')}&page={page}"
        driver.get(url)

        wait.until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, 'div[data-component-type="s-search-result"]')
            )
        )

        time.sleep(2)
        product_cards = driver.find_elements(By.CSS_SELECTOR, 'div[data-component-type="s-search-result"]')

        for card in product_cards:
            title = None
            rating = None
            review_count = None

            # TITLE
            try:
                title = card.find_element(By.CSS_SELECTOR, "h2 span").text.strip()
            except:
                pass

            # RATING - try several possible selectors
            rating_text = None
            selectors_to_try = [
                "span.a-icon-alt",
                "i span.a-icon-alt",
                '[aria-label*="out of 5 stars"]',
                '[aria-label*="stars"]'
            ]

            for sel in selectors_to_try:
                try:
                    el = card.find_element(By.CSS_SELECTOR, sel)
                    rating_text = el.text.strip() or el.get_attribute("aria-label")
                    if rating_text:
                        break
                except:
                    continue

            rating = clean_rating(rating_text)

            try:
                review_tag = card.find_element(
                    By.CSS_SELECTOR,
                    'span.a-size-mini.puis-normal-weight-text.s-underline-text'
                )
                
                review_text = review_tag.text.strip()

                review_text = review_text.replace("(", "").replace(")", "").lower()

                if "k" in review_text:
                    number = float(review_text.replace("k", ""))
                    review_count = int(number * 1000)
                else:
                    review_count = int(review_text.replace(",", ""))

            except:
                pass

            if title:
                rows.append({
                    "title": title,
                    "rating": rating,
                    "number of reviews": review_count,
                    "page": page,
                    "source": "Amazon"
                })

            

        time.sleep(2)

finally:
    driver.quit()

df = pd.DataFrame(rows)
print(df.head())
print(df["rating"].head(20))
df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_FILE}")


                                               title  rating  \
0  Soundcore by Anker Q20i Hybrid Active Noise Ca...     4.6   
1  BERIBES Bluetooth Headphones Over Ear, 65H Pla...     4.5   
2  Soundcore by Anker Q20i Hybrid Active Noise Ca...     4.6   
3  JBL Tune 720BT - Wireless Over-Ear Headphones ...     4.6   
4  JBL Tune 510BT - Bluetooth headphones with up ...     4.5   

   number of reviews  page  source  
0              59800     1  Amazon  
1              53200     1  Amazon  
2              59800     1  Amazon  
3              12700     1  Amazon  
4              90600     1  Amazon  
0     4.6
1     4.5
2     4.6
3     4.6
4     4.5
5     4.7
6     4.3
7     4.5
8     4.6
9     4.6
10    4.5
11    4.8
12    4.4
13    4.2
14    4.6
15    4.6
16    4.8
17    4.5
18    4.8
19    4.4
Name: rating, dtype: float64
Saved 306 rows to amazon_wireless_headphones.csv


In [29]:
df = pd.read_csv("amazon_wireless_headphones.csv")
df.head()

,title,rating,number of reviews,page,source
0,JBL Tune 720BT - Wireless Over-Ear Headphones ...,4.6,12700,1,Amazon
1,JBL Tune 510BT - Bluetooth headphones with up ...,4.5,90600,1,Amazon
2,Beats Studio Pro - Premium Wireless Over-Ear H...,4.5,26900,1,Amazon
3,JBL Tune 510BT - Bluetooth headphones with up ...,4.5,90600,1,Amazon
4,Beats Solo 4 - Wireless On-Ear Bluetooth Headp...,4.6,24900,1,Amazon


In [35]:
df.shape

(306, 5)